## Training Setup
To run these case studies, first make sure you have compiled the necessary OCaml files using the "loss_setup.sh" script in the root of the repository.

Once this is done, begin by running the two code cells below, followed by all the cells in the Unconstrained Model section. Then, to run any case study, run all the cells in each lettered section in order.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import ltl.dmp_train as LTL # DMP training from Innes and Ramamoorthy with our generated loss function
from ltl.constraint import Constraint
from datetime import datetime
import time

USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

def t_stamp():
    return datetime.now().strftime("%Y-%m-%d-%H")

def runtime(t0):
    diff = time.time()-t0
    print("Runtime : {} hr {} min {} sec".format(int(diff//3600),int((diff%3600)//60), int(diff%60)))

In [ ]:
train_set, val_set = LTL.load_data("drive/demos/patrol")

## Unconstrained Model

In [ ]:
t0 = time.time()
baseline_model, baseline_dmp = LTL.create_model_for_train_set(train_set)
baseline_train_losses, baseline_val_losses = LTL.training_loop(baseline_model, baseline_dmp, train_set, val_set, None, False, nu=1.0, N=500, report_step=10)
runtime(t0)
print (baseline_val_losses[-1])

In [ ]:
LTL.plot_vs_sample(baseline_model, baseline_dmp, train_set[0], title="Unconstrained")

In [ ]:
LTL.plot_losses(baseline_train_losses, baseline_val_losses)

## Constrained Case Study

In each of the constrained case studies below, we define the constraint via the Constraint class using 4 parameters.

1. Constraint string
2. A list of trace transformation functions
3. A list of constants
4. Gamma parameter

The constraint string is interpreted using a parser that translates the following, using parentheses to disambiguate order of operation:
1. "n < m" "n <= m" "n == m" "n != m" "n >= m" "n > m": (n, m are integers pointing at an element of the trace) each is turned into a comparison constraint.
2. "! c": Not c.
3. "N c": Next c.
4. "X c": Weak Next c.
5. "G c": Always c.
6. "F c": Eventually c.
7. "c1 U c2": c1 Weak Until c2
8. "c1 R c2": c1 Strong Release c2
9. "c1 & c2": c1 And c2
10. "c1 | c2": c1 Or c2
11. "c1 <-> c2": c1 (If And Only If) c2
12. "c1 -> c2": c1 Implies c2

The list of trace transformation functions act on the x,y coordinate pairs in the trace to produce tensors of float measurements of interest to the constraint, for example, distance from a point or line. Each function is referred to in the constraint string via an integer referring to the position of its output in the list. For example, if [f,g] is the list (f being a function returning two elements per x,y coordinate pair, and g returning one), "1 < 2" returns "the second result of f < the result of g". If "None" is passed, we use the raw x,y coordinates, with "0" referring to x and "1" to y.

The list of constants is a list of floating point numbers used in the constraint. Each constant is referred to in the constraint string via an integer referring to its position in the list + the size of the outputs of trace transformation functions. For example, if [f,g] is the list of trace transformation functions (as above), and [0.5,0.9] is the list of constants, "2 < 4" returns "the results of g < 0.9". If "None" is passed, we do not use any constants.

The gamma parameter is a floating point number corresponding to the smoothing parameter gamma. It is typically 0.005.

### (a) Avoid

In [ ]:
#set the co-ordinates for the point to be avoided, and the distance threshold.

avoid_x = 0.4
avoid_y = 0.4
distance = 0.1

avoid_pt = torch.Tensor([avoid_x,avoid_y]).to(DEVICE)

#define a function that given calculates the distance from the avoid point for any x,y coordinate pair
dist_f = lambda x : torch.sqrt(torch.sum((x-avoid_pt)**2,dim=-1)).unsqueeze(-1) # Dist from avoid_pt

#we modify the results graph to include a coloured circle showing the area we want to avoid
def ann_plt(plt,fig,ax):
    ax.add_artist(plt.Circle((avoid_x, avoid_y), distance, color='r', alpha=0.1))
    plt.scatter([avoid_x],[avoid_y],marker='x',color='r', s=100)

#define the constraint, including the functions and constants needed to modify the trace to calculate the results we need
#note in the constraint definition, "1" refers to the distance constant, and "0" refers to the calculated distance from the avoid point
#thus, the constraint means: ALWAYS (distance constant) <= (distance from avoid point)
constraint = Constraint("G 1 <= 0", # constraint string
                            dist_f, # distance transform
                            [distance], # distance threshold
                            0.005) #gamma

#setup to save the results
results_root = "drive/logs/robot-exps-{}".format(t_stamp())

#train the model and output time taken every 10 iterations for 500 iterations total; then print the results
t0 = time.time()
model, dmp = LTL.create_model_for_train_set(train_set)
train_losses, val_losses = LTL.training_loop(model, dmp, train_set, val_set, constraint, False, nu=1.0, N=500, report_step=10)
runtime(t0)
print (val_losses[-1])

In [ ]:
LTL.plot_models(baseline_model, model, dmp, train_set[0], title="Avoid", ann_f=ann_plt)

In [ ]:
LTL.plot_losses(train_losses, val_losses)

### (b) Until

In [ ]:
#set the x and y thresholds
xthresh = 0.6
ythresh = 0.4

#modify the results graph to put red lines at our thresholds
def ann_plt(plt,fig,ax):
    plt.axhline(ythresh, color='r')
    plt.axvline(xthresh, color='r')

#define the constraint to be used.
#"(y <= 0.4) Until (0.6 <= x)"
constraint = Constraint("(1 <= 2) U (3 <= 0)", # constraint string
                            None, # identity transform
                            [ythresh,xthresh], # consts
                                0.005) # gamma

#setup to save the results
results_root = "drive/logs/robot-exps-{}".format(t_stamp())

#train the model and output time taken every 10 iterations for 500 iterations total; then print the results
t0 = time.time()
model, dmp = LTL.create_model_for_train_set(train_set)
train_losses, val_losses = LTL.training_loop(model, dmp, train_set, val_set, constraint, False, nu=1.0, N=500, report_step=10)
runtime(t0)
print (val_losses[-1])

In [ ]:
LTL.plot_models(baseline_model, model, dmp, train_set[0], title="Until", ann_f=ann_plt)

In [ ]:
LTL.plot_losses(train_losses, val_losses)

### (c) Patrol

In [ ]:
#begin by defining the points to be reached
reach_pt1 = torch.Tensor([0.2,0.4]).to(DEVICE)
reach_pt2 = torch.Tensor([0.85,0.6]).to(DEVICE)

#define a function d that calculates the distance from the reach points for any x,y coordinate pair
def dist_tensor(pt):
    return (lambda x : torch.sqrt(torch.sum((x-pt)**2,dim=-1)).unsqueeze(-1))

d = lambda x : torch.cat((dist_tensor(reach_pt1)(x), dist_tensor(reach_pt2)(x)), dim=-1)

#modify the results graph to put crosses at each of the reach points
def ann_plt(plt,fig,ax):
    plt.scatter([0.2],[0.4],marker='x',color='g', s=100)
    plt.scatter([0.85],[0.6],marker='x',color='g', s=100)

#define the constraint to be used.
#"(Eventually (distance to pt 1 <= 0.0)) And (Eventually (distance to pt 2 <= 0.0))"
constraint = Constraint("(F 0 <= 2) & (F 1 <= 3)", # constraint string
                            d, # distance transform
                            [0.0,0.0], # distance thresholds to pt1 and pt2, respectively
                            0.005) # gamma

#setup to save the results
results_root = "drive/logs/robot-exps-{}".format(t_stamp())

#train the model and output time taken every 10 iterations for 500 iterations total; then print the results
t0 = time.time()
model, dmp = LTL.create_model_for_train_set(train_set)
train_losses, val_losses = LTL.training_loop(model, dmp, train_set, val_set, constraint, False, nu=1.0, N=500, report_step=10)
runtime(t0)
print (val_losses[-1])

In [ ]:
LTL.plot_models(baseline_model, model, dmp, train_set[0], title="Patrol", ann_f=ann_plt)

In [ ]:
LTL.plot_losses(train_losses, val_losses)

### (d) Loop

In [ ]:
#define the points to be reached
reach_pt1 = torch.Tensor([0.2,0.6]).to(DEVICE)
reach_pt2 = torch.Tensor([0.4,0.8]).to(DEVICE)

#define a function d that calculates the distance from the reach points for any x,y coordinate pair
def dist_tensor(pt):
    return (lambda x : torch.sqrt(torch.sum((x-pt)**2,dim=-1)).unsqueeze(-1))

d = lambda x : torch.cat((dist_tensor(reach_pt1)(x), dist_tensor(reach_pt2)(x)), dim=-1)

#modify the results graph to put crosses at each of the reach points
def ann_plt(plt,fig,ax):
    plt.scatter([0.2],[0.6],marker='x',color='g', s=100)
    plt.scatter([0.4],[0.8],marker='x',color='g', s=100)

#define the constraint to be used.
#"Eventually ((distance to pt 1 <= 0.0) And (Weak Next (Eventually (distance to pt 2 <= 0.0))))"
constraint = Constraint("F ((1 <= 2) & (X (F (0 <= 3))))", # constraint string
                            d, # distance transform
                            [0.0,0.0], # distance thresholds to pt1 and pt2, respectively
                            0.005) # gamma

#setup to save the results
results_root = "drive/logs/robot-exps-{}".format(t_stamp())

#train the model and output time taken every 10 iterations for 500 iterations total; then print the results
t0 = time.time()
model, dmp = LTL.create_model_for_train_set(train_set)
train_losses, val_losses = LTL.training_loop(model, dmp, train_set, val_set, constraint, False, nu=1.0, N=500, report_step=10)
runtime(t0)
print (val_losses[-1])

In [ ]:
LTL.plot_models(baseline_model, model, dmp, train_set[0], title="Loop", ann_f=ann_plt)

In [ ]:
LTL.plot_losses(train_losses, val_losses)

### (e) compound test

In [ ]:
#define the points to be reached and avoided
avoid_pt = torch.Tensor([0.5,0.5]).to(DEVICE)
reach_pt = torch.Tensor([0.7,0.5]).to(DEVICE)
dist = 0.1
y_thresh = 0.8

#define a function d that calculates the distance from the reach and avoid points for any x,y coordinate pair, and its y component
def dist_tensor(pt):
    return (lambda x : torch.sqrt(torch.sum((x-pt)**2,dim=-1)).unsqueeze(-1))

def y_tensor():
    return (lambda x : x[:,:,1].unsqueeze(-1))

d = lambda x : torch.cat((dist_tensor(avoid_pt)(x), dist_tensor(reach_pt)(x), y_tensor()(x)), dim=-1)

#modify the results graph to put crosses at each of the reach points, add a red avoidance circle, and put a red line at the y threshold
def ann_plt(plt,fig,ax):
    ax.add_artist(plt.Circle((0.5, 0.5), dist, color='r', alpha=0.1))
    plt.scatter([0.5],[0.5],marker='x',color='g', s=100)
    plt.scatter([0.7],[0.5],marker='x',color='g', s=100)
    plt.axhline(y_thresh, color='r')

#define the constraint to be used.
#"(Always (0.1 <= distance from avoid point)) And (Eventually (distance from reach point <= 0.0)) And (Always (y-coordinate <= 0.8))"
constraint = Constraint("(G (3 <= 0)) & (F (1 <= 4)) & (G (2 <= 5))", # constraint string
                            d, # transform
                            [dist,0.0,y_thresh], # distance thresholds to pt1 and pt2, respectively, and const for y
                            0.005) # gamma

#setup to save the results
results_root = "drive/logs/robot-exps-{}".format(t_stamp())

#train the model and output time taken every 10 iterations for 500 iterations total; then print the results
t0 = time.time()
model, dmp = LTL.create_model_for_train_set(train_set)
train_losses, val_losses = LTL.training_loop(model, dmp, train_set, val_set, constraint, False, nu=1.0, N=500, report_step=10)
runtime(t0)
print (val_losses[-1])

In [ ]:
LTL.plot_models(baseline_model, model, dmp, train_set[0], title="Compound", ann_f=ann_plt)

In [ ]:
LTL.plot_losses(train_losses, val_losses)

### (f) Double Loop

In [ ]:
#define the points to be reached
reach_pt1 = torch.Tensor([0.2,0.6]).to(DEVICE)
reach_pt2 = torch.Tensor([0.4,0.8]).to(DEVICE)
reach_pt3 = torch.Tensor([0.6,0.2]).to(DEVICE)
reach_pt4 = torch.Tensor([0.8,0.4]).to(DEVICE)

#define a function d that calculates the distance from the reach points for any x,y coordinate pair
def dist_tensor(pt):
    return (lambda x : torch.sqrt(torch.sum((x-pt)**2,dim=-1)).unsqueeze(-1))

d = lambda x : torch.cat((dist_tensor(reach_pt1)(x), 
                         dist_tensor(reach_pt2)(x),
                         dist_tensor(reach_pt3)(x),
                         dist_tensor(reach_pt4)(x)),
                         dim=-1)

#modify the results graph to put crosses at each of the reach points
def ann_plt(plt,fig,ax):
        plt.scatter([0.2],[0.6],marker='x',color='g', s=100)
        plt.scatter([0.4],[0.8],marker='x',color='g', s=100)
        plt.scatter([0.6],[0.2],marker='x',color='g', s=100)
        plt.scatter([0.8],[0.4],marker='x',color='g', s=100)
    
#define the constraint to be used.
#"(Eventually ((distance from reach point 2 <= 0.0) And (Weak Next (Eventually (distance from reach point 1 <= 0.0))))) 
#  And (Eventually ((distance from reach point 4 <= 0.0) And (Weak Next (Eventually (distance from reach point 3 <= 0.0)))))"
c_d1 = "0 <= 4"
c_d2 = "1 <= 5"
c_d3 = "2 <= 6"
c_d4 = "3 <= 7"

ltl_constraint = "F ({c_d2} & X (F {c_d1})) & F({c_d4} & X(F {c_d3}))".format(c_d1=c_d1, c_d2=c_d2, c_d3 = c_d3, c_d4 = c_d4)

constraint = Constraint(ltl_constraint, # constraint string
                            d, # distance transform
                            [0.0,0.0,0.0,0.0], # distance thresholds to pt1 and pt2, respectively
                             0.01) # gamma

#setup to save the results
results_root = "drive/logs/robot-exps-{}".format(t_stamp())

#train the model and output time taken every 10 iterations for 500 iterations total; then print the results
t0 = time.time()
model, dmp = LTL.create_model_for_train_set(train_set)
train_losses, val_losses = LTL.training_loop(model, dmp, train_set, val_set, constraint, False, nu=1.0, N=500, report_step=1)
runtime(t0)
print (val_losses[-1])

In [ ]:
LTL.plot_models(baseline_model, model, dmp, train_set[0], title="Double Loop", ann_f=ann_plt)

In [ ]:
LTL.plot_losses(train_losses, val_losses)

(g) Gamma test

In [ ]:
from torch.utils.data import DataLoader

#define the points to be reached and avoided
avoid_pt = torch.Tensor([0.5,0.5]).to(DEVICE)
reach_pt = torch.Tensor([0.7,0.5]).to(DEVICE)
dist = 0.1
y_thresh = 0.8

#define a function d that calculates the distance from the reach and avoid points for any x,y coordinate pair, and its y component
def dist_tensor(pt):
    return (lambda x : torch.sqrt(torch.sum((x-pt)**2,dim=-1)).unsqueeze(-1))

def y_tensor():
    return (lambda x : x[:,:,1].unsqueeze(-1))

d = lambda x : torch.cat((dist_tensor(avoid_pt)(x), dist_tensor(reach_pt)(x), y_tensor()(x)), dim=-1)

#modify the results graph to put crosses at each of the reach points, add a red avoidance circle, and put a red line at the y threshold
def ann_plt(plt,fig,ax):
    ax.add_artist(plt.Circle((0.5, 0.5), dist, color='r', alpha=0.1))
    plt.scatter([0.5],[0.5],marker='x',color='g', s=100)
    plt.scatter([0.7],[0.5],marker='x',color='g', s=100)
    plt.axhline(y_thresh, color='r')

constraint2 = Constraint("(G (3 <= 0)) & (F (1 <= 4)) & (G (2 <= 5))", # constraint string
                        d, # transform
                        [dist,0.0,y_thresh], # distance thresholds to pt1 and pt2, respectively, and const for y
                        0) # gamma

#process failed with gamma = 1
for gamma in [0.5,0.1,0.05,0.01,0.005,0.001]:

    #define the constraint to be used.
    #"(Always (0.1 <= distance from avoid point)) And (Eventually (distance from reach point <= 0.0)) And (Always (y-coordinate <= 0.8))"
    constraint = Constraint("(G (3 <= 0)) & (F (1 <= 4)) & (G (2 <= 5))", # constraint string
                                d, # transform
                                [dist,0.0,y_thresh], # distance thresholds to pt1 and pt2, respectively, and const for y
                                gamma) # gamma
    
    #setup to save the results
    results_root = "drive/logs/robot-exps-{}".format(t_stamp())
    
    #train the model and output time taken every 10 iterations for 500 iterations total; then print the results
    print("Combined constraint, gamma = " + str(gamma))
    t0 = time.time()
    model, dmp = LTL.create_model_for_train_set(train_set)
    train_losses, val_losses = LTL.training_loop(model, dmp, train_set, val_set, constraint, False, nu=1.0, N=500, report_step=10)
    runtime(t0)
    print ("Imitation loss: " + str(val_losses[-1][0]) + "; Constraint loss (gamma): " + str(val_losses[-1][1]) + "; Total loss: " + str(val_losses[-1][2]))
    model.eval()
    learned_weights2 = model(train_set[0][0].unsqueeze(0))
    learned_dmp_rollout2 = dmp.rollout_torch(train_set[0][0][0].unsqueeze(0), train_set[0][0][-1].unsqueeze(0), learned_weights2, 1.0)[0][0].detach().unsqueeze(0).cpu()
    loss = constraint2.loss(learned_dmp_rollout2)
    print ("True loss: " + str(loss.item()) + "; difference: " + str(val_losses[-1][1]-loss.item()))
    LTL.plot_models(baseline_model, model, dmp, train_set[0], title="Combined constraint, gamma = " + str(gamma), ann_f=ann_plt)
    LTL.plot_losses(train_losses, val_losses)